# KYC — Pipeline d'extraction d'identité / Identity Extraction Pipeline
**Domino Data Lab · fully offline · local Qwen VL model**

This notebook implements the complete pipeline:

| Stage | What it does | Output |
|---|---|---|
| 0 | Configuration & environment check | console |
| 1 | Unzip `kyc_documents.zip`, keep **only** the 5 required PDFs | `02_selected_documents/<customer>/` |
| 2 | Presence / absence report of the 5 documents per customer | `01_document_presence_report.csv` |
| 3 | Target customers = those having `JUSTIFICATIF IDENTITE.PDF` | `02_target_customers.csv` |
| 4 | PDF → page images, orientation fix, deskew, conservative enhancement | `03_page_images/` |
| 5 | Local VLM OCR extraction, page by page, strict JSON | `04_results/<customer>.json` |
| 6 | Multi-page merge, MRZ **validation-only**, consolidated reports | `05_reports/` |

**Governing principle (hard-coded throughout):**
> When forced to choose between an incomplete/uncertain result and a plausible but unsupported
> result, **always return the incomplete one**. `null` is the correct answer for an unreadable field.

Nothing in this notebook repairs, completes, normalises or "fixes" an extracted value.
Checksum validation (MRZ) is **reporting only** and never rewrites a character.

## Dependencies

Everything runs offline. Install once into the Domino environment (or bake into the compute
environment image):

```bash
pip install --no-index --find-links /path/to/wheels \
    "transformers>=4.49" accelerate torch \
    pymupdf pypdfium2 pillow numpy pandas opencv-python-headless \
    qwen-vl-utils openpyxl
# optional, only used for page-orientation detection (0/90/180/270):
# apt-get install -y tesseract-ocr && pip install pytesseract
```

* `pymupdf` (or `pypdfium2`) — PDF page rasterisation, no poppler needed.
* `opencv-python-headless` — deskew, CLAHE, quality metrics. Optional: the notebook degrades
  gracefully to PIL-only preprocessing if OpenCV is absent.
* `pytesseract` — **optional**. Only used to detect 90°/180°/270° page rotation. If it is not
  installed the notebook falls back to a VLM orientation probe (a layout question, never a
  field-value question) or to no rotation at all.

In [ ]:
# =========================================================================
# STAGE 0 — CONFIGURATION
# =========================================================================
import os

# Hard offline mode: never let transformers try to reach huggingface.co from Domino.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from pathlib import Path

# -------------------------------------------------------------------------
# 0.1  Input / output paths  --> EDIT THESE
# -------------------------------------------------------------------------
ZIP_PATH = Path("/domino/datasets/local/kyc/kyc_documents.zip")
WORK_DIR = Path("/domino/datasets/local/kyc/run_01")

SELECTED_DIR = WORK_DIR / "02_selected_documents"   # the 5 required PDFs, per customer
IMAGES_DIR   = WORK_DIR / "03_page_images"          # raw + preprocessed page renders
RESULTS_DIR  = WORK_DIR / "04_results"              # one JSON per target customer
REPORTS_DIR  = WORK_DIR / "05_reports"              # CSV / XLSX reports
LOG_DIR      = WORK_DIR / "06_logs"

# -------------------------------------------------------------------------
# 0.2  Local model (Domino ModelHub mount)
# -------------------------------------------------------------------------
MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B/main"

TORCH_DTYPE   = "bfloat16"   # "bfloat16" | "float16" | "float32"
DEVICE_MAP    = "auto"       # "auto" | "cuda:0" | "cpu"
MAX_NEW_TOKENS = 1024
ATTN_IMPL     = None         # e.g. "flash_attention_2" if installed; None = library default

# Deterministic decoding is mandatory here: sampling invents characters.
GEN_KWARGS = dict(do_sample=False, num_beams=1, repetition_penalty=1.0)

MOCK_MODEL = False           # True = run the whole pipeline without the model.
                             # The mock returns an all-null result (it never invents anything).

# -------------------------------------------------------------------------
# 0.3  Rendering & preprocessing
# -------------------------------------------------------------------------
RENDER_DPI          = 300     # 300 dpi is the sweet spot for ID cards / passports
MAX_PAGES_PER_DOC   = 12      # safety cap
MAX_IMAGE_SIDE      = 1600    # px, sent to the VLM (Qwen VL pixel budget)
MIN_IMAGE_SIDE      = 900     # upscale small scans up to this before inference

DO_ORIENTATION      = True    # detect 0/90/180/270
ORIENTATION_METHOD  = "auto"  # "auto" (tesseract OSD -> VLM probe) | "osd" | "vlm" | "none"
DO_DESKEW           = True
MAX_SKEW_DEG        = 12.0    # ignore anything larger, it is layout not skew
DO_ENHANCE          = True    # grayscale + CLAHE + mild edge-preserving denoise ONLY
                              # (no sharpening, no binarisation, no inpainting: those invent pixels)

RETRY_ON_RAW        = True    # if the enhanced page yields nothing, retry on the untouched render
MULTI_IMAGE_MODE    = False   # True = send all pages of a document in ONE prompt
                              # False (default) = one page per call, then merge

# -------------------------------------------------------------------------
# 0.4  Pipeline behaviour
# -------------------------------------------------------------------------
ENABLE_FUZZY_FILENAME_MATCH = True
FUZZY_THRESHOLD   = 0.88      # difflib ratio; fuzzy hits are FLAGGED in the report for review
EXPAND_NESTED_ZIPS = True
RESUME            = True      # skip customers already having a result JSON
MAX_CUSTOMERS     = None      # None = all; set an int for a smoke test
JSON_RETRIES      = 2         # retries when the model does not return parsable JSON

# On a value conflict between two pages of the same document:
#   "null"              -> return null + flag for human review   (default, safest)
#   "highest_confidence"-> keep the highest-confidence value + flag
CONFLICT_POLICY = "null"

for _d in (SELECTED_DIR, IMAGES_DIR, RESULTS_DIR, REPORTS_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

RUN_ID = None  # set in the next cell
print("Work dir :", WORK_DIR)
print("ZIP      :", ZIP_PATH, "| exists:", ZIP_PATH.exists())
print("Model    :", MODEL_PATH, "| exists:", Path(MODEL_PATH).exists())

In [ ]:
# =========================================================================
# 0.5 — IMPORTS & ENVIRONMENT REPORT
# =========================================================================
import io, re, sys, json, time, math, shutil, zipfile, hashlib, difflib
import platform, unicodedata, traceback
from collections import OrderedDict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

Image.MAX_IMAGE_PIXELS = None            # large A4 scans at 300/400 dpi

# ---- optional dependencies, all guarded --------------------------------
try:
    import pymupdf as fitz
except Exception:
    try:
        import fitz
    except Exception:
        fitz = None

try:
    import pypdfium2 as pdfium
except Exception:
    pdfium = None

try:
    import cv2
except Exception:
    cv2 = None

try:
    import pytesseract
    from pytesseract import Output as TessOutput
    pytesseract.get_tesseract_version()
except Exception:
    pytesseract = None
    TessOutput = None

try:
    import torch
except Exception:
    torch = None

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
UTC = lambda: datetime.now(timezone.utc).isoformat()

def env_report():
    rows = [
        ("run_id", RUN_ID),
        ("python", platform.python_version()),
        ("platform", platform.platform()),
        ("numpy", np.__version__),
        ("pandas", pd.__version__),
        ("pymupdf", (getattr(fitz, "__version__", None) or getattr(fitz, "VersionBind", "present"))
                    if fitz else "MISSING"),
        ("pypdfium2", getattr(pdfium, "V_PYPDFIUM2", "present") if pdfium else "MISSING"),
        ("opencv", cv2.__version__ if cv2 else "MISSING (degraded preprocessing)"),
        ("pytesseract", "available" if pytesseract else "MISSING (orientation via VLM probe)"),
        ("torch", torch.__version__ if torch else "MISSING"),
        ("cuda", (torch.cuda.is_available() if torch else False)),
    ]
    if torch is not None and torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            rows.append((f"gpu[{i}]", f"{p.name} · {p.total_memory/1e9:.1f} GB"))
    width = max(len(r[0]) for r in rows)
    for k, v in rows:
        print(f"{k:<{width}} : {v}")

env_report()

if fitz is None and pdfium is None:
    print("\n!! No PDF rasteriser available (pymupdf / pypdfium2). Stage 4 will fail.")

## Stage 1 — Unzip and keep only the 5 required documents

Scanned KYC folders are messy: accents, underscores, mixed case, trailing `(1)`, and the
occasional typo. Filenames are therefore **normalised** (accent-stripped, upper-cased,
punctuation collapsed) and matched against a controlled alias table.

Three match levels are recorded so nothing silently slips through:

* `exact` — normalised name equals a known alias
* `contains` — alias is contained in the normalised name (e.g. `JUSTIFICATIF IDENTITE 001`)
* `fuzzy` — `difflib` ratio ≥ `FUZZY_THRESHOLD`; **always flagged for human review**

Note the source list contains `CARTON SIGNATUTE.PDF` (a typo for *SIGNATURE*). Both spellings are
kept as aliases so folders are matched whichever way the file was named.

In [ ]:
# =========================================================================
# 1.1 — DOCUMENT CATALOG & FILENAME MATCHING
# =========================================================================
ALLOWED_EXT = {".pdf"}

def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFKD", str(text))
                   if not unicodedata.combining(c))

def norm_name(text):
    """Accent-free, upper-case, punctuation-collapsed form used ONLY for matching filenames."""
    s = strip_accents(text).upper()
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

REQUIRED_DOCS = OrderedDict([
    ("JUSTIFICATIF_IDENTITE", {
        "label": "JUSTIFICATIF IDENTITE.PDF",
        "aliases": ["JUSTIFICATIF IDENTITE", "JUSTIFICATIF D IDENTITE", "JUSTIFICATIF DE IDENTITE",
                    "JUSTIF IDENTITE", "PIECE IDENTITE", "PIECE D IDENTITE",
                    "JUSTIFICATIF IDENTITE CLIENT", "IDENTITE"],
    }),
    ("JUSTIFICATIF_DOMICILE", {
        "label": "JUSTIFICATIF DOMICILE.PDF",
        "aliases": ["JUSTIFICATIF DOMICILE", "JUSTIFICATIF DE DOMICILE", "JUSTIF DOMICILE",
                    "PREUVE DE DOMICILE", "DOMICILE"],
    }),
    ("CONVENTION_COMPTE", {
        "label": "CONVENTION COMPTE.PDF",
        "aliases": ["CONVENTION COMPTE", "CONVENTION DE COMPTE", "CONVENTION DU COMPTE",
                    "CONVENTION OUVERTURE COMPTE"],
    }),
    ("FATCA", {
        "label": "FATCA.PDF",
        "aliases": ["FATCA", "FORMULAIRE FATCA", "FATCA CRS", "AUTOCERTIFICATION FATCA", "W 9", "W 8BEN"],
    }),
    ("CARTON_SIGNATURE", {
        # the source specification spells it SIGNATUTE; both spellings accepted
        "label": "CARTON SIGNATUTE.PDF",
        "aliases": ["CARTON SIGNATUTE", "CARTON SIGNATURE", "CARTON DE SIGNATURE",
                    "SPECIMEN SIGNATURE", "SPECIMEN DE SIGNATURE", "CARTON SIGNATURES"],
    }),
])

for _k, _spec in REQUIRED_DOCS.items():
    _spec["norm_aliases"] = sorted({norm_name(a) for a in ([_spec["label"]] + _spec["aliases"])},
                                   key=len, reverse=True)

DOC_KEYS = list(REQUIRED_DOCS.keys())
ID_DOC_KEY = "JUSTIFICATIF_IDENTITE"


def match_required_doc(filename):
    """Return (doc_key, match_type, score). doc_key is None when the file is not required."""
    p = Path(str(filename))
    if p.suffix.lower() not in ALLOWED_EXT:
        return None, "wrong_extension", 0.0

    stem = norm_name(p.stem)
    stem = re.sub(r"\s+\d+$", "", stem).strip()          # drop trailing copy counters "... 1"

    for key, spec in REQUIRED_DOCS.items():              # 1. exact
        if stem in spec["norm_aliases"]:
            return key, "exact", 1.0

    for key, spec in REQUIRED_DOCS.items():              # 2. containment
        for alias in spec["norm_aliases"]:
            if len(alias) >= 5 and alias in stem:
                return key, "contains", 0.95

    if ENABLE_FUZZY_FILENAME_MATCH:                      # 3. fuzzy (flagged)
        best_key, best_score = None, 0.0
        for key, spec in REQUIRED_DOCS.items():
            for alias in spec["norm_aliases"]:
                r = difflib.SequenceMatcher(None, stem, alias).ratio()
                if r > best_score:
                    best_key, best_score = key, r
        if best_score >= FUZZY_THRESHOLD:
            return best_key, "fuzzy", round(best_score, 4)

    return None, "no_match", 0.0


# quick self-test of the matcher
for _t in ["JUSTIFICATIF IDENTITE.PDF", "justificatif_identité (1).pdf", "CARTON SIGNATUTE.PDF",
           "Carton-Signature.PDF", "FATCA.pdf", "RIB.pdf", "convention de compte v2.pdf",
           "photo.jpg"]:
    print(f"{_t:<38} -> {match_required_doc(_t)}")

In [ ]:
# =========================================================================
# 1.2 — SAFE ZIP EXTRACTION (only the required documents are written to disk)
# =========================================================================
def _safe_relpath(member_name):
    """Reject absolute paths, drive letters and .. traversal (zip-slip)."""
    name = member_name.replace("\\", "/")
    if name.startswith("/") or re.match(r"^[A-Za-z]:", name):
        return None
    parts = [p for p in name.split("/") if p not in ("", ".")]
    if any(p == ".." for p in parts):
        return None
    return parts


def _detect_root_prefix(all_parts):
    """If every member sits under one single wrapper folder, strip it."""
    roots = {p[0] for p in all_parts if len(p) > 1}
    if len(roots) == 1:
        root = roots.pop()
        deeper = [p for p in all_parts if len(p) > 2 and p[0] == root]
        if deeper:
            return root
    return None


def _iter_zip_entries(zip_path):
    """Yield (member_path_parts, filename, size, reader_fn). Handles one level of nested ZIPs."""
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.is_dir()]
        parts_list, kept = [], []
        for info in infos:
            parts = _safe_relpath(info.filename)
            if parts is None:
                print("  ! unsafe member skipped:", info.filename)
                continue
            parts_list.append(parts)
            kept.append(info)
        root = _detect_root_prefix(parts_list)
        for info, parts in zip(kept, parts_list):
            if root and parts[0] == root:
                parts = parts[1:]
            if not parts:
                continue
            if EXPAND_NESTED_ZIPS and parts[-1].lower().endswith(".zip"):
                try:
                    inner_bytes = zf.read(info)
                    with zipfile.ZipFile(io.BytesIO(inner_bytes)) as izf:
                        for iinfo in izf.infolist():
                            if iinfo.is_dir():
                                continue
                            iparts = _safe_relpath(iinfo.filename)
                            if iparts is None:
                                continue
                            combined = parts[:-1] + iparts
                            payload = izf.read(iinfo)
                            yield combined, combined[-1], iinfo.file_size, (lambda b=payload: b)
                    continue
                except Exception as exc:
                    print("  ! nested zip unreadable:", "/".join(parts), exc)
            yield parts, parts[-1], info.file_size, (lambda i=info: zf.read(i))


def extract_required_documents(zip_path, out_dir):
    """Walk the archive, copy ONLY the 5 required PDFs, and catalogue everything seen."""
    assert Path(zip_path).exists(), f"ZIP not found: {zip_path}"
    out_dir = Path(out_dir)
    all_rows, sel_rows = [], []
    seen_targets = {}

    for parts, fname, size, read_fn in _iter_zip_entries(zip_path):
        customer_id = parts[0] if len(parts) > 1 else "_ARCHIVE_ROOT_"
        doc_key, match_type, score = match_required_doc(fname)

        all_rows.append(dict(customer_id=customer_id, member="/".join(parts), filename=fname,
                             ext=Path(fname).suffix.lower(), size_bytes=int(size),
                             matched_doc_key=doc_key or "", match_type=match_type,
                             match_score=score))
        if doc_key is None:
            continue

        dest_dir = out_dir / customer_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        key = (customer_id, doc_key)
        n_prev = seen_targets.get(key, 0)
        seen_targets[key] = n_prev + 1
        dest = dest_dir / (f"{doc_key}.pdf" if n_prev == 0 else f"{doc_key}__dup{n_prev+1}.pdf")

        data = read_fn()
        dest.write_bytes(data)
        sel_rows.append(dict(customer_id=customer_id, doc_key=doc_key,
                             source_member="/".join(parts), source_filename=fname,
                             extracted_path=str(dest), match_type=match_type,
                             match_score=score, size_bytes=len(data),
                             is_duplicate=bool(n_prev),
                             sha256=hashlib.sha256(data).hexdigest()))

    return pd.DataFrame(sel_rows), pd.DataFrame(all_rows)


t0 = time.time()
selected_df, all_members_df = extract_required_documents(ZIP_PATH, SELECTED_DIR)
print(f"Archive walked in {time.time()-t0:.1f}s")
print(f"  members seen        : {len(all_members_df)}")
print(f"  customers (folders) : {all_members_df['customer_id'].nunique()}")
print(f"  required PDFs kept  : {len(selected_df)}")
if not selected_df.empty:
    print(f"  fuzzy matches       : {(selected_df['match_type'] == 'fuzzy').sum()}  <-- review these")
    display(selected_df.head(10))

## Stage 2 — Presence / absence report

One row per customer folder, one column per required document (`PRESENT` / `MISSING`),
plus the columns a reviewer needs: number of copies found, fuzzy-matched filenames,
and the total number of files in the folder.

In [ ]:
# =========================================================================
# 2.1 — BUILD THE PRESENCE / ABSENCE REPORT
# =========================================================================
def build_inventory(all_members_df, selected_df):
    customers = sorted(set(all_members_df["customer_id"]) | set(selected_df.get("customer_id", [])))
    by_cust_doc = {}
    if not selected_df.empty:
        for (cid, dk), grp in selected_df.groupby(["customer_id", "doc_key"]):
            by_cust_doc[(cid, dk)] = grp

    rows, long_rows = [], []
    for cid in customers:
        files_in_folder = all_members_df[all_members_df["customer_id"] == cid]
        row = OrderedDict(customer_id=cid, n_files_in_folder=len(files_in_folder))
        n_present, fuzzy_flags = 0, []
        for dk in DOC_KEYS:
            grp = by_cust_doc.get((cid, dk))
            present = grp is not None and len(grp) > 0
            row[dk] = "PRESENT" if present else "MISSING"
            row[f"{dk}__n_copies"] = int(len(grp)) if present else 0
            row[f"{dk}__source_filename"] = grp.iloc[0]["source_filename"] if present else ""
            row[f"{dk}__match_type"] = grp.iloc[0]["match_type"] if present else ""
            if present:
                n_present += 1
                if (grp["match_type"] == "fuzzy").any():
                    fuzzy_flags.append(dk)
            long_rows.append(dict(customer_id=cid, document=REQUIRED_DOCS[dk]["label"],
                                  doc_key=dk, status="PRESENT" if present else "MISSING",
                                  n_copies=int(len(grp)) if present else 0,
                                  source_filename=grp.iloc[0]["source_filename"] if present else "",
                                  match_type=grp.iloc[0]["match_type"] if present else ""))
        row["n_required_present"] = n_present
        row["n_required_missing"] = len(DOC_KEYS) - n_present
        row["folder_complete"] = (n_present == len(DOC_KEYS))
        row["is_target_customer"] = (row[ID_DOC_KEY] == "PRESENT")
        row["fuzzy_matched_docs"] = ";".join(fuzzy_flags)
        row["needs_filename_review"] = bool(fuzzy_flags)
        rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(long_rows)


inventory_df, inventory_long_df = build_inventory(all_members_df, selected_df)

inv_csv = REPORTS_DIR / "01_document_presence_report.csv"
inventory_df.to_csv(inv_csv, index=False, encoding="utf-8-sig")
inventory_long_df.to_csv(REPORTS_DIR / "01b_document_presence_long.csv", index=False, encoding="utf-8-sig")
(REPORTS_DIR / "01_document_presence_report.json").write_text(
    inventory_df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
all_members_df.to_csv(REPORTS_DIR / "00_archive_file_catalog.csv", index=False, encoding="utf-8-sig")
try:
    with pd.ExcelWriter(REPORTS_DIR / "01_document_presence_report.xlsx") as xw:
        inventory_df.to_excel(xw, sheet_name="presence", index=False)
        inventory_long_df.to_excel(xw, sheet_name="detail", index=False)
except Exception as exc:
    print("(xlsx skipped:", exc, ")")

print("Presence report ->", inv_csv)
print()
summary = pd.DataFrame({
    "PRESENT": [(inventory_df[k] == "PRESENT").sum() for k in DOC_KEYS],
    "MISSING": [(inventory_df[k] == "MISSING").sum() for k in DOC_KEYS],
}, index=[REQUIRED_DOCS[k]["label"] for k in DOC_KEYS])
summary["coverage_%"] = (100 * summary["PRESENT"] / max(len(inventory_df), 1)).round(1)
display(summary)
display(inventory_df[["customer_id"] + DOC_KEYS + ["n_required_present", "folder_complete"]].head(15))

## Stage 3 — Target customers

Target = the customer folder contains `JUSTIFICATIF IDENTITE.PDF`. Only these folders go through
the OCR stage; everything else is reported as missing and left untouched.

In [ ]:
# =========================================================================
# 3.1 — TARGET CUSTOMER LIST
# =========================================================================
target_df = inventory_df.loc[inventory_df[ID_DOC_KEY] == "PRESENT",
                             ["customer_id", f"{ID_DOC_KEY}__source_filename",
                              f"{ID_DOC_KEY}__match_type", f"{ID_DOC_KEY}__n_copies"]].copy()
target_df.columns = ["customer_id", "source_filename", "match_type", "n_copies"]
target_df["id_pdf_path"] = target_df["customer_id"].map(
    lambda c: str(SELECTED_DIR / c / f"{ID_DOC_KEY}.pdf"))
target_df = target_df.sort_values("customer_id").reset_index(drop=True)

target_df.to_csv(REPORTS_DIR / "02_target_customers.csv", index=False, encoding="utf-8-sig")
(REPORTS_DIR / "02_target_customers.json").write_text(
    json.dumps(target_df["customer_id"].tolist(), ensure_ascii=False, indent=2), encoding="utf-8")

non_target = inventory_df.loc[inventory_df[ID_DOC_KEY] != "PRESENT", "customer_id"].tolist()
(REPORTS_DIR / "02b_customers_without_identity_doc.json").write_text(
    json.dumps(non_target, ensure_ascii=False, indent=2), encoding="utf-8")

TARGET_CUSTOMERS = target_df["customer_id"].tolist()
if MAX_CUSTOMERS:
    TARGET_CUSTOMERS = TARGET_CUSTOMERS[:MAX_CUSTOMERS]
    print(f"(MAX_CUSTOMERS={MAX_CUSTOMERS} -> smoke-test subset)")

print(f"Target customers      : {len(target_df)}")
print(f"Without identity doc  : {len(non_target)}")
display(target_df.head(10))

## Stage 4 — PDF → images, orientation, deskew, conservative enhancement

Scans are rotated, skewed, noisy and low-contrast. The preprocessing chain is deliberately
**conservative**: it repositions and equalises, it never invents pixel detail.

| Step | Used | Deliberately NOT used |
|---|---|---|
| Rasterise at 300 dpi | ✔ | — |
| Rotate by 0/90/180/270 (OSD or VLM probe) | ✔ | — |
| Deskew ≤ 12° (bicubic, replicated border) | ✔ | — |
| Grayscale + CLAHE (contrast equalisation) | ✔ | — |
| Bilateral denoise (edge preserving) | ✔ | Median/Gaussian blur (eats thin strokes) |
| Lanczos upscale of small scans | ✔ | Super-resolution / GAN upscaling (**hallucinates glyphs**) |
| — | — | Unsharp mask, binarisation, inpainting, "document restoration" |

Both the **raw** render and the **processed** image are written to disk so any extracted
character can be audited against the original pixels.

In [ ]:
# =========================================================================
# 4.1 — PDF PAGE RASTERISATION
# =========================================================================
def render_pdf_pages(pdf_path, dpi=None, max_pages=None):
    """PDF -> list of RGB PIL images. PyMuPDF first, pypdfium2 as fallback."""
    dpi = dpi or RENDER_DPI
    max_pages = max_pages or MAX_PAGES_PER_DOC
    pdf_path = str(pdf_path)
    images = []

    if fitz is not None:
        doc = fitz.open(pdf_path)
        try:
            for i, page in enumerate(doc):
                if i >= max_pages:
                    break
                pix = page.get_pixmap(dpi=dpi, colorspace=fitz.csRGB, alpha=False)
                images.append(Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy())
        finally:
            doc.close()
        return images

    if pdfium is not None:
        pdf = pdfium.PdfDocument(pdf_path)
        try:
            for i in range(min(len(pdf), max_pages)):
                images.append(pdf[i].render(scale=dpi / 72.0).to_pil().convert("RGB"))
        finally:
            pdf.close()
        return images

    raise RuntimeError("No PDF rasteriser available: install pymupdf or pypdfium2.")


def pdf_page_count(pdf_path):
    if fitz is not None:
        d = fitz.open(str(pdf_path))
        n = d.page_count
        d.close()
        return n
    if pdfium is not None:
        p = pdfium.PdfDocument(str(pdf_path))
        n = len(p)
        p.close()
        return n
    return -1

In [ ]:
# =========================================================================
# 4.2 — IMAGE QUALITY METRICS  (measured, never used to "fix" anything)
# =========================================================================
def to_gray_np(img):
    g = np.asarray(img.convert("L"), dtype=np.uint8)
    return g


def quality_metrics(img):
    g = to_gray_np(img)
    h, w = g.shape
    if cv2 is not None:
        blur = float(cv2.Laplacian(g, cv2.CV_64F).var())
    else:
        gx = np.diff(g.astype(np.float32), axis=1)
        gy = np.diff(g.astype(np.float32), axis=0)
        blur = float(gx.var() + gy.var())
    m = dict(
        width=int(w), height=int(h), megapixels=round(w * h / 1e6, 2),
        blur_laplacian_var=round(blur, 1),
        contrast_std=round(float(g.std()), 1),
        brightness_mean=round(float(g.mean()), 1),
        dark_ratio=round(float((g < 40).mean()), 3),
        bright_ratio=round(float((g > 215).mean()), 3),
    )
    flags = []
    if min(w, h) < 700:              flags.append("low_resolution")
    if m["blur_laplacian_var"] < 100: flags.append("blurred")
    if m["contrast_std"] < 35:        flags.append("low_contrast")
    if m["brightness_mean"] < 70:     flags.append("under_exposed")
    if m["brightness_mean"] > 205:    flags.append("over_exposed")
    if m["bright_ratio"] > 0.85:      flags.append("washed_out")
    m["quality_flags"] = flags
    m["quality_grade"] = ("poor" if len(flags) >= 2 else "fair" if flags else "good")
    return m

In [ ]:
# =========================================================================
# 4.3 — ORIENTATION, DESKEW, CONSERVATIVE ENHANCEMENT
# =========================================================================
def detect_orientation_osd(img):
    """Tesseract OSD -> degrees to rotate COUNTER-clockwise to make text upright, or None."""
    if pytesseract is None:
        return None, None
    try:
        small = img.copy()
        small.thumbnail((1500, 1500))
        osd = pytesseract.image_to_osd(small, output_type=TessOutput.DICT,
                                       config="--psm 0 -c min_characters_to_try=5")
        return int(osd.get("rotate", 0)) % 360, float(osd.get("orientation_conf", 0.0))
    except Exception:
        return None, None


def estimate_skew_angle(img, max_deg=None):
    """Small-angle skew from the dominant text-block orientation. Returns degrees (float)."""
    max_deg = MAX_SKEW_DEG if max_deg is None else max_deg
    if cv2 is None:
        return 0.0
    g = to_gray_np(img)
    scale = 1200.0 / max(g.shape)
    if scale < 1.0:
        g = cv2.resize(g, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    thr = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    thr = cv2.dilate(thr, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3)), iterations=1)
    coords = cv2.findNonZero(thr)
    if coords is None or len(coords) < 50:
        return 0.0
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle += 90
    elif angle > 45:
        angle -= 90
    return 0.0 if abs(angle) > max_deg else float(round(angle, 2))


def rotate_small_angle(img, angle):
    if abs(angle) < 0.1:
        return img
    if cv2 is None:
        return img.rotate(-angle, resample=Image.BICUBIC, expand=True, fillcolor=(255, 255, 255))
    a = np.asarray(img)
    h, w = a.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    out = cv2.warpAffine(a, M, (nw, nh), flags=cv2.INTER_CUBIC,
                         borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(out)


def enhance_for_ocr(img):
    """Grayscale + CLAHE + mild edge-preserving denoise. No sharpening, no binarisation."""
    if cv2 is None:
        return ImageOps.autocontrast(img.convert("L")).convert("RGB"), ["pil_autocontrast"]
    g = to_gray_np(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g = clahe.apply(g)
    g = cv2.bilateralFilter(g, d=5, sigmaColor=45, sigmaSpace=45)
    return Image.fromarray(g).convert("RGB"), ["grayscale", "clahe", "bilateral_denoise"]


def fit_side(img, max_side=None, min_side=None):
    max_side = MAX_IMAGE_SIDE if max_side is None else max_side
    min_side = MIN_IMAGE_SIDE if min_side is None else min_side
    w, h = img.size
    ops = []
    if max(w, h) > max_side:
        s = max_side / max(w, h)
        img = img.resize((max(1, int(w * s)), max(1, int(h * s))), Image.LANCZOS)
        ops.append(f"downscale_{s:.2f}")
    elif min(img.size) < min_side:
        s = min(2.0, min_side / min(img.size))          # cap the upscale: no invented detail
        w, h = img.size
        img = img.resize((int(w * s), int(h * s)), Image.LANCZOS)
        ops.append(f"upscale_lanczos_{s:.2f}")
    return img, ops


def preprocess_page(img, orient_fn=None):
    """Returns (processed_image, metadata). Metadata records every operation applied."""
    meta = {"ops": [], "rotation_deg": 0, "skew_deg": 0.0,
            "orientation_method": "none", "orientation_conf": None}

    if DO_ORIENTATION and ORIENTATION_METHOD != "none":
        deg, conf = (None, None)
        if ORIENTATION_METHOD in ("auto", "osd"):
            deg, conf = detect_orientation_osd(img)
            if deg is not None:
                meta["orientation_method"] = "tesseract_osd"
        if deg is None and ORIENTATION_METHOD in ("auto", "vlm") and orient_fn is not None:
            try:
                deg = orient_fn(img)
                meta["orientation_method"] = "vlm_probe"
            except Exception as exc:
                meta["orientation_error"] = str(exc)
                deg = None
        if deg:
            img = img.rotate(deg, expand=True)          # PIL rotates counter-clockwise
            meta["rotation_deg"] = int(deg) % 360
            meta["ops"].append(f"rotate_{int(deg) % 360}")
        meta["orientation_conf"] = conf

    if DO_DESKEW:
        ang = estimate_skew_angle(img)
        if abs(ang) >= 0.3:
            img = rotate_small_angle(img, ang)
            meta["skew_deg"] = ang
            meta["ops"].append(f"deskew_{ang:+.2f}")

    if DO_ENHANCE:
        img, ops = enhance_for_ocr(img)
        meta["ops"].extend(ops)

    img, ops = fit_side(img)
    meta["ops"].extend(ops)
    meta["final_size"] = list(img.size)
    return img, meta

## Stage 5 — Local VLM

### 5.1 The prompts

The system prompt and the per-image prompt below are reproduced **verbatim** from the
specification. They are hashed into every result file so that any extraction can be traced back
to the exact instruction text that produced it.

In [ ]:
# =========================================================================
# 5.1 — PROMPTS (verbatim, hashed for auditability)
# =========================================================================
SYSTEM_PROMPT = """You are a high-precision OCR and document extraction engine specialized in identity documents such as passports, national identity cards, residence permits, and driver's licenses.

CRITICAL ANTI-HALLUCINATION RULES

NEVER guess a character, digit, word, name, date, or document number.
NEVER infer missing characters from context.
NEVER complete partially visible text.
NEVER correct spelling, transliteration, formatting, or apparent OCR errors.
NEVER use external knowledge to reconstruct unreadable text.
NEVER assume what a field "should" contain based on the document type.
If a character cannot be distinguished reliably from the image, mark that character as uncertain.
If a complete field cannot be read reliably, return null.
A partially unreadable value is preferable to an invented complete value.
Do NOT manufacture a value merely because the requested JSON field requires a value.
Do NOT answer with explanations, guesses, or conversational text.
The image is the ONLY authoritative source of the extracted value.

VISUAL EVIDENCE RULE

For every extracted value, ask internally:
"Can I directly see every character of this value in the image?"
If the answer is NO:
Do not guess.
Do not infer.
Do not reconstruct.
Return the value as null or mark the uncertain character(s).

For example, if the image appears to contain:
A12?45B7
and the fourth character cannot be reliably distinguished, DO NOT output:
A12345B7
Instead output:
{
"value": null,
"status": "uncertain",
"uncertain_positions": [4]
}

CHARACTER-LEVEL TRANSCRIPTION

Preserve exactly what is visually present.
Do not:
fix spelling
normalize names
translate names
expand abbreviations
change Arabic names into a preferred Latin spelling
replace visually similar characters unless the image clearly establishes the character

Pay particular attention to visually similar characters:
0 / O
1 / I / L
2 / Z
5 / S
6 / G
8 / B
C / G
U / V
D / O
M / N

If the image does not allow a reliable distinction, mark the character as uncertain.

DOCUMENT FIELDS

Extract only the fields that are requested.
Typical fields may include:
surname
given_names
date_of_birth
place_of_birth
nationality
sex
document_number
issue_date
expiry_date
issuing_authority
personal_number
MRZ

Do not invent fields that are not visible.

MRZ

If a passport contains a Machine Readable Zone:
Transcribe the MRZ exactly as visible.
Do not reconstruct missing characters.
Preserve < characters.
Do not "repair" the MRZ simply because the expected ICAO format suggests another character.
If a character is unreadable, mark it as uncertain.
If the MRZ is too degraded to reliably transcribe, return null.
If MRZ validation is performed, validation may identify an inconsistency but must NEVER be used to invent the missing character.

IMAGE QUALITY

Before extraction, evaluate:
resolution
blur
compression artifacts
contrast
skew
cropping
shadows
missing portions
character visibility

If image quality prevents reliable extraction, report that instead of guessing.
Do not assume that image enhancement makes an unreadable character readable.

FIELD-LEVEL CONFIDENCE

For each field, assign one of:
high: every character is clearly visible
medium: most characters are visible but one or more are somewhat ambiguous
low: significant ambiguity exists
unreadable: the value cannot be reliably extracted

Confidence refers to VISUAL EVIDENCE, not how plausible the resulting value appears.
A plausible value with weak visual evidence must NOT receive high confidence.

OUTPUT REQUIREMENT

Return ONLY valid JSON.

Never return Markdown.
Never return explanations.
Never return commentary before or after the JSON.

Use this structure:

{
"document_type": {"value": null, "confidence": "unreadable"},
"surname": {"value": null, "confidence": "unreadable"},
"given_names": {"value": null, "confidence": "unreadable"},
"date_of_birth": {"value": null, "confidence": "unreadable"},
"place_of_birth": {"value": null, "confidence": "unreadable"},
"nationality": {"value": null, "confidence": "unreadable"},
"sex": {"value": null, "confidence": "unreadable"},
"document_number": {"value": null, "confidence": "unreadable"},
"issue_date": {"value": null, "confidence": "unreadable"},
"expiry_date": {"value": null, "confidence": "unreadable"},
"issuing_authority": {"value": null, "confidence": "unreadable"},
"mrz": {"value": null, "confidence": "unreadable"},
"image_quality": {"overall": "unreadable", "reason": null}
}

MOST IMPORTANT PRINCIPLE

When forced to choose between:
A. returning an incomplete/uncertain result
and
B. returning a plausible but unsupported result
ALWAYS choose A.

False information is worse than missing information.

The correct behavior for an unreadable document is:
null
not a guess.

The model must behave like a forensic transcription system, not like a conversational assistant."""


USER_PROMPT = """Extract the identity information from this document according to the OCR rules in your system instructions.

Read the image directly.

Do not infer or reconstruct anything that is not clearly visible.

For every requested field:

Locate the corresponding field in the document.
Read the characters directly from the pixels.
Verify every character visually.
If one or more characters cannot be reliably distinguished, do not guess them.
If the field cannot be reliably read, return null.
Assign confidence based strictly on visual evidence.

Pay particular attention to:

document number
dates
names
visually similar letters and digits
MRZ characters
characters damaged by blur, compression, low contrast, or scanning artifacts.

Return ONLY the JSON object defined in the system instructions."""


RETRY_SUFFIX = ("\n\nYour previous answer was not valid JSON. Return ONLY the JSON object defined "
                "in the system instructions: no Markdown fences, no commentary, no explanation.")

PROMPT_HASHES = {
    "system_sha256": hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
    "user_sha256": hashlib.sha256(USER_PROMPT.encode("utf-8")).hexdigest(),
}
print("prompt hashes:", json.dumps(PROMPT_HASHES, indent=2))
print("system prompt chars:", len(SYSTEM_PROMPT), "| user prompt chars:", len(USER_PROMPT))

### 5.2 Loading the local model

The loader is deliberately generic: it reads `config.json` from the mounted ModelHub path and
tries the matching vision-language class, falling back through
`AutoModelForImageTextToText` → `AutoModelForVision2Seq` → `AutoModelForCausalLM`.

> **Check before running the batch.** The path must point at a **vision-language** checkpoint
> (Qwen2-VL / Qwen2.5-VL / Qwen3-VL family). A text-only Qwen checkpoint cannot read an image and
> the loader will say so explicitly rather than silently producing text-only guesses.
> Set `MOCK_MODEL = True` to rehearse the whole pipeline without a GPU.

In [ ]:
# =========================================================================
# 5.2 — LOCAL VLM WRAPPER
# =========================================================================
VISION_HINTS = ("vl", "vision", "image_text", "qwen2_vl", "qwen2_5_vl", "qwen3_vl",
                "llava", "internvl", "idefics", "pixtral")


class MockVLM:
    """Pipeline rehearsal without a GPU. Returns the all-null schema: it invents nothing."""
    name = "MOCK"

    def generate(self, images, system_prompt, user_prompt, max_new_tokens=None):
        return json.dumps({
            **{f: {"value": None, "confidence": "unreadable"} for f in
               ["document_type", "surname", "given_names", "date_of_birth", "place_of_birth",
                "nationality", "sex", "document_number", "issue_date", "expiry_date",
                "issuing_authority", "mrz"]},
            "image_quality": {"overall": "unreadable", "reason": "MOCK_MODEL enabled"},
        })


class LocalVLM:
    def __init__(self, model_path, dtype=TORCH_DTYPE, device_map=DEVICE_MAP):
        from transformers import AutoConfig, AutoProcessor
        self.model_path = str(model_path)
        assert Path(self.model_path).exists(), f"Model path not found: {self.model_path}"
        assert torch is not None, "PyTorch is required to load the model."

        self.config = AutoConfig.from_pretrained(self.model_path, trust_remote_code=True,
                                                 local_files_only=True)
        arch = " ".join(getattr(self.config, "architectures", []) or []).lower()
        mtype = str(getattr(self.config, "model_type", "")).lower()
        self.is_vision = any(h in arch or h in mtype for h in VISION_HINTS) or \
                         hasattr(self.config, "vision_config")
        if not self.is_vision:
            raise RuntimeError(
                f"'{self.model_path}' does not look like a vision-language checkpoint "
                f"(model_type={mtype!r}, architectures={getattr(self.config, 'architectures', None)}). "
                "OCR from images requires a VL model (e.g. Qwen2.5-VL). "
                "Point MODEL_PATH at the VL checkpoint in ModelHub, or set MOCK_MODEL=True to "
                "rehearse the pipeline.")

        self.processor = AutoProcessor.from_pretrained(self.model_path, trust_remote_code=True,
                                                       local_files_only=True)
        torch_dtype = getattr(torch, dtype) if isinstance(dtype, str) else dtype
        kwargs = dict(torch_dtype=torch_dtype, device_map=device_map,
                      trust_remote_code=True, local_files_only=True, low_cpu_mem_usage=True)
        if ATTN_IMPL:
            kwargs["attn_implementation"] = ATTN_IMPL

        self.model, self.loader = None, None
        errors = []
        candidates = []
        try:
            import transformers as _tf
            for cls_name in (getattr(self.config, "architectures", []) or []):
                if hasattr(_tf, cls_name):
                    candidates.append((cls_name, getattr(_tf, cls_name)))
            for cls_name in ("AutoModelForImageTextToText", "AutoModelForVision2Seq",
                             "AutoModelForCausalLM"):
                if hasattr(_tf, cls_name):
                    candidates.append((cls_name, getattr(_tf, cls_name)))
        except Exception as exc:
            errors.append(f"transformers import: {exc}")

        for cls_name, cls in candidates:
            try:
                self.model = cls.from_pretrained(self.model_path, **kwargs)
                self.loader = cls_name
                break
            except Exception as exc:
                errors.append(f"{cls_name}: {type(exc).__name__}: {exc}")
        if self.model is None:
            raise RuntimeError("Could not load the model.\n  " + "\n  ".join(errors))

        self.model.eval()
        gc = getattr(self.model, "generation_config", None)
        if gc is not None:
            gc.do_sample = False
            gc.temperature = None
            gc.top_p = None
            gc.top_k = None
        self.name = f"{Path(self.model_path).parent.name} [{self.loader}]"
        print(f"Loaded {self.name} · dtype={dtype} · device_map={device_map}")

    # -------------------------------------------------------------------
    def _build_inputs(self, images, system_prompt, user_prompt):
        content = [{"type": "image", "image": im} for im in images]
        content.append({"type": "text", "text": user_prompt})
        messages = [{"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                    {"role": "user", "content": content}]
        text = self.processor.apply_chat_template(messages, tokenize=False,
                                                  add_generation_prompt=True)
        try:
            from qwen_vl_utils import process_vision_info
            image_inputs, video_inputs = process_vision_info(messages)
        except Exception:
            image_inputs, video_inputs = list(images), None
        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt")
        return {k: (v.to(self.model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

    def generate(self, images, system_prompt, user_prompt, max_new_tokens=None):
        images = [images] if isinstance(images, Image.Image) else list(images)
        inputs = self._build_inputs(images, system_prompt, user_prompt)
        with torch.inference_mode():
            out = self.model.generate(**inputs,
                                      max_new_tokens=max_new_tokens or MAX_NEW_TOKENS,
                                      **GEN_KWARGS)
        trimmed = [o[len(i):] for i, o in zip(inputs["input_ids"], out)]
        decoded = self.processor.batch_decode(trimmed, skip_special_tokens=True,
                                              clean_up_tokenization_spaces=False)
        return decoded[0].strip()


t0 = time.time()
if MOCK_MODEL:
    vlm = MockVLM()
    print("MOCK_MODEL = True -> no real inference, all fields will be null.")
else:
    vlm = LocalVLM(MODEL_PATH)
print(f"Model ready in {time.time()-t0:.1f}s")

In [ ]:
# =========================================================================
# 5.3 — VLM ORIENTATION PROBE (layout question only, never a field value)
# =========================================================================
ORIENTATION_SYSTEM = ("You judge the physical orientation of a scanned page. "
                      "You answer with a single number and nothing else.")
ORIENTATION_USER = ("By how many degrees counter-clockwise must this scanned page be rotated so "
                    "that the printed text reads normally left-to-right and upright? "
                    "Answer with exactly one of these numbers and nothing else: 0, 90, 180, 270.")


def make_vlm_orientation_fn(model):
    """Returns fn(PIL image) -> degrees in {0,90,180,270}. Falls back to 0 on any doubt."""
    def _fn(img):
        probe = img.copy()
        probe.thumbnail((768, 768))
        raw = model.generate([probe], ORIENTATION_SYSTEM, ORIENTATION_USER, max_new_tokens=8)
        m = re.search(r"\b(0|90|180|270)\b", raw or "")
        return int(m.group(1)) if m else 0
    return _fn


ORIENT_FN = make_vlm_orientation_fn(vlm) if ORIENTATION_METHOD in ("auto", "vlm") else None

if not DO_ORIENTATION or ORIENTATION_METHOD == "none":
    _strategy = "disabled"
else:
    _osd = pytesseract is not None and ORIENTATION_METHOD in ("auto", "osd")
    _strategy = ("tesseract OSD -> VLM probe" if (_osd and ORIENT_FN) else
                 "tesseract OSD" if _osd else
                 "VLM probe" if ORIENT_FN else
                 "disabled (no OSD backend available)")
print("Orientation strategy:", _strategy)

### 5.4 Strict JSON contract

The model's answer is parsed defensively and coerced onto a fixed schema. The coercion is
**subtractive only**:

* a missing field becomes `null` / `unreadable` — it is never filled in;
* an unexpected field is moved to `extra_fields_ignored`, never merged into a real field;
* a non-`null` value arriving without a confidence is downgraded to `low`, never promoted;
* `{"value": null, "status": "uncertain", "uncertain_positions": [...]}` (the shape the prompt
  asks for on ambiguous characters) is preserved in the field metadata;
* if the response cannot be parsed after `JSON_RETRIES`, the page result is an all-`null`
  record with `parse_error` — the pipeline fails **closed**, never open.

In [ ]:
# =========================================================================
# 5.4 — JSON EXTRACTION, VALIDATION, COERCION
# =========================================================================
FIELDS = ["document_type", "surname", "given_names", "date_of_birth", "place_of_birth",
          "nationality", "sex", "document_number", "issue_date", "expiry_date",
          "issuing_authority", "mrz"]
OPTIONAL_FIELDS = ["personal_number"]          # captured if the model returns it, never required
CORE_FIELDS = ["surname", "given_names", "date_of_birth", "document_number"]
CONF_LEVELS = ("high", "medium", "low", "unreadable")
CONF_RANK = {"high": 3, "medium": 2, "low": 1, "unreadable": 0}
NULLISH = {"", "null", "none", "n/a", "na", "unreadable", "not visible", "not readable", "-", "--"}


def blank_result(reason=None):
    r = {f: {"value": None, "confidence": "unreadable"} for f in FIELDS}
    r["image_quality"] = {"overall": "unreadable", "reason": reason}
    return r


def extract_json_block(text):
    """Pull the first balanced {...} object out of the raw model output."""
    if not text:
        return None
    t = text.strip()
    t = re.sub(r"^```[a-zA-Z]*\s*", "", t)
    t = re.sub(r"\s*```$", "", t).strip()
    start = t.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        c = t[i]
        if in_str:
            if esc:            esc = False
            elif c == "\\":    esc = True
            elif c == '"':     in_str = False
            continue
        if c == '"':           in_str = True
        elif c == "{":         depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return t[start:i + 1]
    return None


def _clean_value(raw, field=None):
    """Normalise the CONTAINER only. The characters themselves are never modified."""
    meta = {}
    if isinstance(raw, dict):
        for k in ("status", "uncertain_positions", "note"):
            if k in raw:
                meta[k] = raw[k]
        raw = raw.get("value", None)
    if isinstance(raw, (list, tuple)):
        sep = "\n" if field == "mrz" else " "
        raw = sep.join(str(x) for x in raw if x is not None)
    if raw is None:
        return None, meta
    if isinstance(raw, bool):
        return None, meta
    if isinstance(raw, (int, float)):
        raw = str(raw)
    if not isinstance(raw, str):
        return None, meta
    s = raw.strip()
    if s.lower() in NULLISH:
        return None, meta
    return s, meta


def coerce_result(obj):
    """Map arbitrary model output onto the fixed schema. Subtractive only."""
    out, warnings = {}, []
    if not isinstance(obj, dict):
        r = blank_result("model did not return a JSON object")
        r["_warnings"] = ["not_a_json_object"]
        return r

    lower_map = {str(k).strip().lower(): k for k in obj.keys()}
    consumed = set()

    for f in FIELDS + OPTIONAL_FIELDS:
        src_key = lower_map.get(f)
        if src_key is None and f == "mrz":
            src_key = lower_map.get("machine_readable_zone")
        if src_key is None:
            if f in FIELDS:
                out[f] = {"value": None, "confidence": "unreadable"}
                warnings.append(f"missing_field:{f}")
            continue
        consumed.add(src_key)
        node = obj[src_key]
        value, vmeta = _clean_value(node, field=f)
        conf = node.get("confidence") if isinstance(node, dict) else None
        conf = str(conf).strip().lower() if conf is not None else None
        if conf not in CONF_LEVELS:
            if conf is not None:
                warnings.append(f"bad_confidence:{f}:{conf}")
            conf = "low" if value is not None else "unreadable"   # downgrade, never promote
        if value is None:
            conf = "unreadable"                                    # null can only be unreadable
        entry = {"value": value, "confidence": conf}
        if vmeta:
            entry.update(vmeta)
        out[f] = entry

    iq_key = lower_map.get("image_quality")
    if iq_key is not None:
        consumed.add(iq_key)
        iq = obj[iq_key]
        if isinstance(iq, dict):
            overall = str(iq.get("overall", "unreadable")).strip().lower()
            out["image_quality"] = {
                "overall": overall if overall in ("good", "fair", "poor", "unreadable") else "unreadable",
                "reason": iq.get("reason"),
            }
        else:
            out["image_quality"] = {"overall": "unreadable", "reason": str(iq)}
    else:
        out["image_quality"] = {"overall": "unreadable", "reason": None}
        warnings.append("missing_field:image_quality")

    extra = {k: obj[k] for k in obj if k not in consumed}
    if extra:
        out["extra_fields_ignored"] = extra
        warnings.append("extra_fields:" + ",".join(sorted(map(str, extra)))[:200])
    if warnings:
        out["_warnings"] = warnings
    return out


def parse_model_output(raw_text):
    """Return (coerced_result, parse_ok, error_message)."""
    block = extract_json_block(raw_text)
    if block is None:
        return blank_result("no JSON object in model output"), False, "no_json_found"
    try:
        obj = json.loads(block)
    except json.JSONDecodeError as exc:
        try:                                    # tolerate trailing commas only
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except Exception:
            return blank_result("invalid JSON in model output"), False, f"json_decode_error: {exc}"
    return coerce_result(obj), True, None


# --- self test -----------------------------------------------------------
_demo = """```json
{"document_type":{"value":"PASSPORT","confidence":"high"},
 "surname":{"value":"BEN ALI","confidence":"high"},
 "document_number":{"value":null,"status":"uncertain","uncertain_positions":[4],"confidence":"low"},
 "colour_of_eyes":{"value":"BROWN","confidence":"high"}}
```"""
_r, _ok, _err = parse_model_output(_demo)
print("parse_ok:", _ok, "| err:", _err)
print(json.dumps({k: _r[k] for k in ["surname", "document_number", "date_of_birth",
                                     "image_quality", "_warnings"]}, indent=2, ensure_ascii=False))

In [ ]:
# =========================================================================
# 5.5 — MRZ CHECK DIGITS  (VALIDATION ONLY — never repairs a character)
# =========================================================================
_MRZ_WEIGHTS = (7, 3, 1)


def _mrz_char_value(c):
    if c == "<":
        return 0
    if c.isdigit():
        return int(c)
    if "A" <= c.upper() <= "Z":
        return ord(c.upper()) - 55
    return None


def mrz_check_digit(s):
    total = 0
    for i, c in enumerate(s):
        v = _mrz_char_value(c)
        if v is None:
            return None                       # unknown glyph -> no verdict, no invention
        total += v * _MRZ_WEIGHTS[i % 3]
    return str(total % 10)


def validate_mrz(mrz_value):
    """Report-only ICAO 9303 check-digit verification. Returns a verdict dict, never a new value."""
    if not mrz_value:
        return {"status": "absent", "format": None, "checks": {}}
    lines = [ln.strip().replace(" ", "") for ln in str(mrz_value).splitlines() if ln.strip()]
    if not lines:
        return {"status": "absent", "format": None, "checks": {}}
    lens = [len(ln) for ln in lines]
    fmt = ("TD3" if len(lines) == 2 and max(lens) >= 43 else
           "TD2" if len(lines) == 2 and max(lens) >= 35 else
           "TD1" if len(lines) == 3 else "unknown")
    checks = {}
    try:
        if fmt == "TD3":
            l2 = lines[1]
            spans = {"document_number": (0, 9, 9), "date_of_birth": (13, 19, 19),
                     "expiry_date": (21, 27, 27)}
            for name, (a, b, cd) in spans.items():
                if len(l2) > cd:
                    computed = mrz_check_digit(l2[a:b])
                    printed = l2[cd]
                    checks[name] = {"printed": printed, "computed": computed,
                                    "match": (computed is not None and computed == printed)}
        elif fmt == "TD1":
            l1, l2 = lines[0], lines[1]
            if len(l1) >= 15:
                checks["document_number"] = {"printed": l1[14],
                                             "computed": mrz_check_digit(l1[5:14])}
            if len(l2) >= 7:
                checks["date_of_birth"] = {"printed": l2[6], "computed": mrz_check_digit(l2[0:6])}
            if len(l2) >= 15:
                checks["expiry_date"] = {"printed": l2[14], "computed": mrz_check_digit(l2[8:14])}
            for c in checks.values():
                c["match"] = (c["computed"] is not None and c["computed"] == c["printed"])
    except Exception as exc:
        return {"status": "error", "format": fmt, "checks": {}, "error": str(exc)}

    if not checks:
        status = "not_verifiable"
    elif all(c.get("match") for c in checks.values()):
        status = "valid"
    else:
        status = "mismatch"          # FLAG ONLY: the transcription is left exactly as read
    return {"status": status, "format": fmt, "line_lengths": lens, "checks": checks,
            "note": "validation is informational; no character is ever corrected"}


print(validate_mrz("P<FRADUPONT<<JEAN<<<<<<<<<<<<<<<<<<<<<<<<<<<\n" + "12AB45678" + "9FRA8001015M3001011<<<<<<<<<<<<<<02")["status"])

In [ ]:
# =========================================================================
# 5.6 — PER-IMAGE EXTRACTION RUNNER
# =========================================================================
def run_extraction_on_images(model, images, retries=None):
    """Send image(s) to the VLM and return a strictly-validated result dict."""
    retries = JSON_RETRIES if retries is None else retries
    attempts, raw_text, err = 0, "", None
    user_prompt = USER_PROMPT
    t0 = time.time()
    while attempts <= retries:
        attempts += 1
        try:
            raw_text = model.generate(images, SYSTEM_PROMPT, user_prompt)
        except Exception as exc:
            err = f"inference_error: {type(exc).__name__}: {exc}"
            result = blank_result(err)
            return {"result": result, "parse_ok": False, "error": err, "attempts": attempts,
                    "raw_output": "", "latency_s": round(time.time() - t0, 2)}
        result, ok, err = parse_model_output(raw_text)
        if ok:
            return {"result": result, "parse_ok": True, "error": None, "attempts": attempts,
                    "raw_output": raw_text, "latency_s": round(time.time() - t0, 2)}
        user_prompt = USER_PROMPT + RETRY_SUFFIX      # stricter reminder, identical rules

    # fail closed: no JSON after all retries -> all-null record
    return {"result": blank_result(f"unparsable model output ({err})"), "parse_ok": False,
            "error": err, "attempts": attempts, "raw_output": raw_text,
            "latency_s": round(time.time() - t0, 2)}


def is_empty_result(result):
    return all(result.get(f, {}).get("value") in (None, "") for f in CORE_FIELDS)

### 5.7 Multi-page merge

An identity document is normally scanned over several pages (recto, verso, sometimes several
documents). Each page is extracted independently, then merged field by field:

1. discard `null` / `unreadable` candidates;
2. keep the candidates carrying the **highest confidence**;
3. if those agree → that value wins, with the list of source pages;
4. if they **disagree** → `CONFLICT_POLICY` decides. Default `"null"`: the field is emptied and
   both readings are written to the conflicts report for a human to arbitrate.

Values are compared using a case/spacing-insensitive key, but the **stored value is always the
original transcription** — comparison never rewrites anything.

In [ ]:
# =========================================================================
# 5.7 — MULTI-PAGE MERGE
# =========================================================================
def _compare_key(value):
    """Comparison key ONLY. Never stored, never returned as a value."""
    s = strip_accents(str(value)).upper()
    return re.sub(r"[^A-Z0-9<]+", "", s)


def merge_page_results(page_entries, policy=None):
    """page_entries: list of dicts with keys page_id + result. Returns (merged, conflicts)."""
    policy = CONFLICT_POLICY if policy is None else policy
    merged, conflicts = {}, []

    for f in FIELDS + OPTIONAL_FIELDS:
        cands = []
        for e in page_entries:
            node = (e.get("result") or {}).get(f)
            if not isinstance(node, dict):
                continue
            val, conf = node.get("value"), node.get("confidence", "unreadable")
            if val is None or CONF_RANK.get(conf, 0) == 0:
                continue
            cands.append({"page_id": e["page_id"], "value": val, "confidence": conf,
                          "rank": CONF_RANK.get(conf, 0),
                          "uncertain_positions": node.get("uncertain_positions"),
                          "status": node.get("status")})
        if not cands:
            if f in FIELDS:
                merged[f] = {"value": None, "confidence": "unreadable", "source_pages": []}
            continue

        top_rank = max(c["rank"] for c in cands)
        top = [c for c in cands if c["rank"] == top_rank]
        distinct = {}
        for c in top:
            distinct.setdefault(_compare_key(c["value"]), []).append(c)

        if len(distinct) == 1:
            winner = top[0]
            entry = {"value": winner["value"],
                     "confidence": winner["confidence"],
                     "source_pages": [c["page_id"] for c in top]}
            if winner.get("uncertain_positions"):
                entry["uncertain_positions"] = winner["uncertain_positions"]
            if winner.get("status"):
                entry["status"] = winner["status"]
            merged[f] = entry
        else:
            conflicts.append({"field": f,
                              "candidates": [{"page_id": c["page_id"], "value": c["value"],
                                              "confidence": c["confidence"]} for c in cands]})
            if policy == "highest_confidence":
                winner = top[0]
                merged[f] = {"value": winner["value"], "confidence": "low",
                             "source_pages": [winner["page_id"]],
                             "status": "conflicting_pages", "needs_human_review": True}
            else:   # "null" -> false information is worse than missing information
                merged[f] = {"value": None, "confidence": "unreadable",
                             "source_pages": [], "status": "conflicting_pages",
                             "needs_human_review": True}

    qualities = [(e.get("result") or {}).get("image_quality", {}).get("overall", "unreadable")
                 for e in page_entries]
    order = {"good": 3, "fair": 2, "poor": 1, "unreadable": 0}
    best = max(qualities, key=lambda q: order.get(q, 0)) if qualities else "unreadable"
    reasons = [(e.get("result") or {}).get("image_quality", {}).get("reason")
               for e in page_entries]
    merged["image_quality"] = {"overall": best,
                               "reason": next((r for r in reasons if r), None),
                               "per_page": dict(zip([e["page_id"] for e in page_entries], qualities))}
    return merged, conflicts

## Stage 6 — Batch run over the target customers

Per customer: render → preprocess → extract page by page → merge → validate MRZ → write JSON.
The loop is **resumable** (`RESUME = True` skips customers that already have a result file) and
never aborts the batch on a single failure — errors are captured per customer and reported.

In [ ]:
# =========================================================================
# 6.1 — PER-CUSTOMER PROCESSING
# =========================================================================
def process_customer(customer_id, pdf_path, model, orient_fn=None, save_images=True):
    started = time.time()
    out_img_dir = IMAGES_DIR / customer_id
    if save_images:
        out_img_dir.mkdir(parents=True, exist_ok=True)

    record = {
        "customer_id": customer_id,
        "source_pdf": str(pdf_path),
        "source_sha256": hashlib.sha256(Path(pdf_path).read_bytes()).hexdigest(),
        "run_id": RUN_ID,
        "extracted_at_utc": UTC(),
        "model": {"path": MODEL_PATH if not MOCK_MODEL else "MOCK",
                  "name": getattr(model, "name", "unknown"),
                  "generation": GEN_KWARGS, "max_new_tokens": MAX_NEW_TOKENS},
        "prompt_hashes": PROMPT_HASHES,
        "pipeline": {"render_dpi": RENDER_DPI, "orientation": DO_ORIENTATION,
                     "deskew": DO_DESKEW, "enhance": DO_ENHANCE,
                     "multi_image_mode": MULTI_IMAGE_MODE, "conflict_policy": CONFLICT_POLICY},
        "pages": [], "errors": [],
    }

    pages = render_pdf_pages(pdf_path)
    record["n_pages_rendered"] = len(pages)
    if not pages:
        record["errors"].append("no page rendered from the PDF")
        record["extraction"] = blank_result("no page rendered")
        record["conflicts"] = []
        record["mrz_validation"] = validate_mrz(None)
        record["duration_s"] = round(time.time() - started, 2)
        return record

    prepared = []
    for idx, page in enumerate(pages, start=1):
        raw_q = quality_metrics(page)
        proc, pmeta = preprocess_page(page, orient_fn=orient_fn)
        proc_q = quality_metrics(proc)
        if save_images:
            raw_small, _ = fit_side(page)
            raw_small.save(out_img_dir / f"page_{idx:02d}_raw.png")
            proc.save(out_img_dir / f"page_{idx:02d}_processed.png")
        prepared.append({"index": idx, "raw": page, "processed": proc,
                         "raw_quality": raw_q, "processed_quality": proc_q, "preprocess": pmeta})

    page_entries = []
    if MULTI_IMAGE_MODE:
        run = run_extraction_on_images(model, [p["processed"] for p in prepared])
        page_entries.append({"page_id": "all_pages", **run})
        record["pages"].append({
            "page_id": "all_pages", "pages_included": [p["index"] for p in prepared],
            "parse_ok": run["parse_ok"], "error": run["error"], "attempts": run["attempts"],
            "latency_s": run["latency_s"], "result": run["result"],
            "preprocess": [p["preprocess"] for p in prepared],
            "quality_raw": [p["raw_quality"] for p in prepared]})
    else:
        for p in prepared:
            pid = f"p{p['index']}"
            run = run_extraction_on_images(model, [p["processed"]])
            page_entries.append({"page_id": pid, **run})
            record["pages"].append({
                "page_id": pid, "variant": "processed", "parse_ok": run["parse_ok"],
                "error": run["error"], "attempts": run["attempts"], "latency_s": run["latency_s"],
                "preprocess": p["preprocess"], "quality_raw": p["raw_quality"],
                "quality_processed": p["processed_quality"], "result": run["result"]})

            # second look at the untouched render when enhancement produced nothing
            if RETRY_ON_RAW and is_empty_result(run["result"]):
                raw_img, _ = fit_side(p["raw"])
                run2 = run_extraction_on_images(model, [raw_img])
                pid2 = f"{pid}_raw"
                page_entries.append({"page_id": pid2, **run2})
                record["pages"].append({
                    "page_id": pid2, "variant": "raw", "parse_ok": run2["parse_ok"],
                    "error": run2["error"], "attempts": run2["attempts"],
                    "latency_s": run2["latency_s"], "quality_raw": p["raw_quality"],
                    "result": run2["result"]})

    merged, conflicts = merge_page_results(page_entries)
    record["extraction"] = merged
    record["conflicts"] = conflicts
    record["mrz_validation"] = validate_mrz(merged.get("mrz", {}).get("value"))
    record["parse_errors"] = [e["page_id"] for e in page_entries if not e["parse_ok"]]
    record["needs_human_review"] = bool(
        conflicts or record["parse_errors"]
        or record["mrz_validation"]["status"] == "mismatch"
        or any(merged.get(f, {}).get("confidence") in ("low", "unreadable") for f in CORE_FIELDS))
    record["duration_s"] = round(time.time() - started, 2)
    return record

In [ ]:
# =========================================================================
# 6.2 — BATCH LOOP (resumable, fault tolerant)
# =========================================================================
run_log, failures = [], []
t_start = time.time()

for i, cid in enumerate(TARGET_CUSTOMERS, start=1):
    out_json = RESULTS_DIR / f"{cid}.json"
    if RESUME and out_json.exists():
        print(f"[{i}/{len(TARGET_CUSTOMERS)}] {cid}: skipped (already done)")
        run_log.append({"customer_id": cid, "status": "skipped", "duration_s": 0.0})
        continue

    pdf_path = SELECTED_DIR / cid / f"{ID_DOC_KEY}.pdf"
    try:
        if not pdf_path.exists():
            raise FileNotFoundError(str(pdf_path))
        rec = process_customer(cid, pdf_path, vlm, orient_fn=ORIENT_FN)
        out_json.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        filled = sum(1 for f in FIELDS if rec["extraction"].get(f, {}).get("value") is not None)
        print(f"[{i}/{len(TARGET_CUSTOMERS)}] {cid}: {rec['n_pages_rendered']}p · "
              f"{filled}/{len(FIELDS)} fields · {rec['duration_s']}s"
              f"{' · REVIEW' if rec['needs_human_review'] else ''}")
        run_log.append({"customer_id": cid, "status": "ok", "pages": rec["n_pages_rendered"],
                        "fields_filled": filled, "needs_review": rec["needs_human_review"],
                        "duration_s": rec["duration_s"]})
    except Exception as exc:
        tb = traceback.format_exc()
        print(f"[{i}/{len(TARGET_CUSTOMERS)}] {cid}: FAILED — {type(exc).__name__}: {exc}")
        failures.append({"customer_id": cid, "error": f"{type(exc).__name__}: {exc}",
                         "traceback": tb})
        run_log.append({"customer_id": cid, "status": "failed", "duration_s": 0.0})
        (LOG_DIR / f"{cid}.error.txt").write_text(tb, encoding="utf-8")

elapsed = time.time() - t_start
print(f"\nBatch finished in {elapsed/60:.1f} min · "
      f"ok={sum(1 for r in run_log if r['status']=='ok')} · "
      f"skipped={sum(1 for r in run_log if r['status']=='skipped')} · "
      f"failed={len(failures)}")
pd.DataFrame(run_log).to_csv(LOG_DIR / f"run_log_{RUN_ID}.csv", index=False, encoding="utf-8-sig")
if failures:
    pd.DataFrame(failures)[["customer_id", "error"]].to_csv(
        LOG_DIR / f"failures_{RUN_ID}.csv", index=False, encoding="utf-8-sig")

In [ ]:
# =========================================================================
# 6.3 — CONSOLIDATED REPORTS
# =========================================================================
def flatten_record(rec):
    row = OrderedDict(customer_id=rec["customer_id"],
                      n_pages=rec.get("n_pages_rendered", 0),
                      source_pdf=Path(rec.get("source_pdf", "")).name)
    ex = rec.get("extraction", {})
    for f in FIELDS:
        node = ex.get(f, {}) or {}
        row[f] = node.get("value")
        row[f + "__confidence"] = node.get("confidence", "unreadable")
        row[f + "__pages"] = ";".join(node.get("source_pages", []) or [])
        if node.get("uncertain_positions"):
            row[f + "__uncertain_positions"] = ";".join(map(str, node["uncertain_positions"]))
    row["image_quality"] = (ex.get("image_quality", {}) or {}).get("overall")
    row["mrz_checksum_status"] = (rec.get("mrz_validation", {}) or {}).get("status")
    row["n_conflicts"] = len(rec.get("conflicts", []))
    row["parse_error_pages"] = ";".join(rec.get("parse_errors", []) or [])
    row["needs_human_review"] = rec.get("needs_human_review")
    row["duration_s"] = rec.get("duration_s")
    row["extracted_at_utc"] = rec.get("extracted_at_utc")
    return row


records = []
for p in sorted(RESULTS_DIR.glob("*.json")):
    try:
        records.append(json.loads(p.read_text(encoding="utf-8")))
    except Exception as exc:
        print("unreadable result file:", p.name, exc)

if records:
    results_df = pd.DataFrame([flatten_record(r) for r in records])
    results_df.to_csv(REPORTS_DIR / "03_identity_extraction.csv", index=False, encoding="utf-8-sig")
    (REPORTS_DIR / "03_identity_extraction.json").write_text(
        json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

    conf_rows = []
    for f in FIELDS:
        vc = results_df[f + "__confidence"].value_counts()
        conf_rows.append({"field": f,
                          **{lvl: int(vc.get(lvl, 0)) for lvl in CONF_LEVELS},
                          "filled": int(results_df[f].notna().sum()),
                          "fill_rate_%": round(100 * results_df[f].notna().mean(), 1)})
    confidence_df = pd.DataFrame(conf_rows)
    confidence_df.to_csv(REPORTS_DIR / "04_field_confidence_summary.csv", index=False,
                         encoding="utf-8-sig")

    conflict_rows = []
    for r in records:
        for c in r.get("conflicts", []):
            for cand in c["candidates"]:
                conflict_rows.append({"customer_id": r["customer_id"], "field": c["field"],
                                      "page_id": cand["page_id"], "value": cand["value"],
                                      "confidence": cand["confidence"]})
    if conflict_rows:
        pd.DataFrame(conflict_rows).to_csv(REPORTS_DIR / "05_page_conflicts.csv", index=False,
                                           encoding="utf-8-sig")

    review_df = results_df[results_df["needs_human_review"] == True]
    review_df.to_csv(REPORTS_DIR / "06_needs_human_review.csv", index=False, encoding="utf-8-sig")

    try:
        with pd.ExcelWriter(REPORTS_DIR / "KYC_extraction_report.xlsx") as xw:
            inventory_df.to_excel(xw, sheet_name="1_presence", index=False)
            target_df.to_excel(xw, sheet_name="2_targets", index=False)
            results_df.to_excel(xw, sheet_name="3_extraction", index=False)
            confidence_df.to_excel(xw, sheet_name="4_confidence", index=False)
            review_df.to_excel(xw, sheet_name="5_review", index=False)
    except Exception as exc:
        print("(xlsx skipped:", exc, ")")

    print("=" * 68)
    print(f"customers processed   : {len(results_df)}")
    print(f"needing human review  : {int(results_df['needs_human_review'].sum())}")
    print(f"pages with parse error: {int((results_df['parse_error_pages'] != '').sum())}")
    print(f"MRZ checksum mismatch : {int((results_df['mrz_checksum_status'] == 'mismatch').sum())}")
    print("=" * 68)
    display(confidence_df)
    display(results_df[["customer_id", "document_type", "surname", "given_names",
                        "document_number", "document_number__confidence",
                        "needs_human_review"]].head(20))
else:
    print("No result file found — run the batch cell first.")

print("\nAll outputs under:", WORK_DIR)
for p in sorted(REPORTS_DIR.glob("*")):
    print("  -", p.name)

In [ ]:
# =========================================================================
# 6.4 — SINGLE-DOCUMENT DEBUG VIEW (optional)
# =========================================================================
def inspect(customer_id, page=1):
    """Show the processed page next to the JSON extracted from it."""
    from IPython.display import display as _d
    rec_path = RESULTS_DIR / f"{customer_id}.json"
    if not rec_path.exists():
        print("no result for", customer_id)
        return
    rec = json.loads(rec_path.read_text(encoding="utf-8"))
    img_path = IMAGES_DIR / customer_id / f"page_{page:02d}_processed.png"
    if img_path.exists():
        im = Image.open(img_path)
        im.thumbnail((900, 900))
        _d(im)
    for p in rec["pages"]:
        if p["page_id"] in (f"p{page}", "all_pages"):
            print(json.dumps({"page_id": p["page_id"], "preprocess": p.get("preprocess"),
                              "quality_raw": p.get("quality_raw"),
                              "result": p["result"]}, ensure_ascii=False, indent=2))
            break
    print("\n--- merged ---")
    print(json.dumps(rec["extraction"], ensure_ascii=False, indent=2))
    print("\n--- mrz validation (report only) ---")
    print(json.dumps(rec["mrz_validation"], ensure_ascii=False, indent=2))


# inspect(TARGET_CUSTOMERS[0])

## Operating notes

**Model path.** `MODEL_PATH` must point at a **vision-language** checkpoint in ModelHub
(Qwen2-VL / Qwen2.5-VL / Qwen3-VL). The loader inspects `config.json` and refuses to start on a
text-only checkpoint instead of producing plausible-looking output with no pixels behind it.
`MOCK_MODEL = True` rehearses the full pipeline with no GPU.

**Determinism.** `do_sample=False, num_beams=1`. Sampling is what makes a model invent a digit
that "looks right"; it stays off. Re-running the same page gives the same transcription.

**Multi-language documents.** Nothing in the chain translates or transliterates. Arabic, Latin and
mixed-script values are stored exactly as transcribed, in UTF-8. CSVs are written with a BOM
(`utf-8-sig`) so Excel opens them without mangling accents or Arabic.

**Where the anti-hallucination rules are actually enforced** — not only in the prompt:

| Risk | Guard |
|---|---|
| Model invents a plausible value | prompt + `do_sample=False` |
| Malformed JSON silently dropped | fail closed → all-null record + `parse_error` |
| Missing field silently filled | `coerce_result` writes `null`/`unreadable` |
| Confidence inflated | unknown confidence downgraded to `low`, never promoted |
| `null` value with a high confidence | forced to `unreadable` |
| Pages disagree | `CONFLICT_POLICY="null"` + conflicts report |
| MRZ checksum fails | flagged only, never repaired |
| Enhancement invents glyphs | no sharpening/binarisation/super-resolution; upscale capped at ×2 |
| Unverifiable output | raw + processed page images kept for every extraction |

**Auditability.** Every result JSON carries the source PDF SHA-256, the prompt hashes, the model
name, the generation parameters, the preprocessing operations applied to each page, and the raw
per-page results before merging.

**Throughput.** Roughly 3–8 s per page on one A100 at `MAX_IMAGE_SIDE=1600`. For large batches,
raise `MAX_IMAGE_SIDE` only where quality flags indicate small text, and run several Domino
workers over disjoint slices of `TARGET_CUSTOMERS` — `RESUME=True` makes the slices safe to
re-run.

**Recommended review workflow.** Treat `06_needs_human_review.csv` as the operator queue: it
collects low-confidence core fields, page conflicts, JSON parse failures and MRZ mismatches. Every
row there points at page images in `03_page_images/<customer>/` for side-by-side verification.